### 1. configuração inicial
importação das bibliotecas e inicialização do cliente groq.

In [ ]:
import os
import re 
from dotenv import load_dotenv 
from groq import Groq

# carrega as variáveis do arquivo .env
load_dotenv()

# inicializa o cliente do groq
client = Groq(
    api_key=os.getenv("GROQ_API_KEY")
)

### 2. a classe do agente
estrutura base do agente, responsável por armazenar o contexto da conversa e fazer as requisições ao modelo llama 3.

In [ ]:
class Agent:
    def __init__(self, client, system=""):
        self.client = client
        self.system = system
        self.messages = [] 
        
        if self.system:
            self.messages.append({"role": "system","content": system})
        
    def __call__(self, message=""):
        if message:
            self.messages.append({"role": "user", "content": message})
        
        result = self.execute()
        self.messages.append({"role": "assistant", "content": result})
        return result

    def execute(self):
        completion = self.client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=self.messages
        )
        return completion.choices[0].message.content

### 3. ferramentas e system prompt
resolvi ser conservador quanto a minha implementação e criei uma espécie de "agente de viagens", cujo retornaria o tempo de voo e o clima de determinada cidade que estiver mockada nas ferramentas incluídas. O system prompt foi fortemente baseado no que foi apresentado na aula, mas com alterações de forma a se encaixar no contexto da especifidade do meu agente (e em português, é claro).

def obter_clima(cidade):
    if cidade.lower() == "paris":
        return "15°C e chuvoso"
    elif cidade.lower() == "toquio":
        return "22°C e ensolarado"
    else:
        return "Clima desconhecido"

def tempo_de_voo(destino):
    if destino.lower() == "paris":
        return "11 horas"
    elif destino.lower() == "toquio":
        return "24 horas"
    else:
        return "Tempo de voo desconhecido"
    
system_prompt = """
Você é um agente de viagens e roda em um ciclo de Thought, Action, PAUSE, Observation.
No final, você deve fornecer uma Answer (Resposta).

Use Thought (Pensamento) para descrever o que você precisa fazer.
Use Action (Ação) para rodar uma das ferramentas disponíveis e retorne PAUSE.
Observation (Observação) será o resultado da sua ação.

Suas ferramentas disponíveis são:

obter_clima:
exemplo: obter_clima: Paris
Retorna como está o tempo na cidade.

tempo_de_voo:
exemplo: tempo_de_voo: Toquio
Retorna o tempo de voo saindo do Brasil.

Exemplo de uso:
Question: Como está o clima em Paris e quanto tempo demora para chegar?
Thought: Preciso primeiro descobrir o clima em Paris.
Action: obter_clima: Paris
PAUSE

Você será chamado novamente com:
Observation: 15°C e chuvoso

Thought: Agora preciso saber o tempo de voo para Paris.
Action: tempo_de_voo: Paris
PAUSE

Você será chamado novamente com:
Observation: 11 horas

Answer: O clima em Paris é de 15°C e chuvoso, e o voo saindo do Brasil demora cerca de 11 horas.

Agora é a sua vez:
"""

### 4. execução e teste
o loop principal que gerencia o ciclo react. ao final, testamos o agente com uma pergunta que exige o uso das duas ferramentas criadas.

In [ ]:
def executar_agente(question):
    agente = Agent(client=client, system=system_prompt)
    tools = ["obter_clima", "tempo_de_voo"] 

    next_prompt = question

    for i in range(10):
        result = agente(next_prompt)
        print(result)

        if "PAUSE" in result and "Action" in result:
            bora = re.search(r"Action: ([a-z_]+): (.*)", result, re.IGNORECASE)

            if bora:
                tool = bora.group(1)
                platao = bora.group(2).strip()

                if tool in tools:
                    result_ferramenta = eval(f"{tool}('{platao}')")
                    next_prompt = f"Observation: {result_ferramenta}" 
                    continue

        if "Answer" in result:
            break

pergunta = "qual a temperatura de toquio? E quanto tempo demoro pra chegar em paris?"
print(f"Pergunta: {pergunta}\n")

executar_agente(pergunta)